# Customer Intelligence & Support Copilot
## End-to-end project workflow

**Scenario:** synthetic grocery quick-commerce data. This notebook emphasizes *how data is fetched, inspected, cleaned, validated, stored, and then consumed by downstream AI/ML components*.

### Prerequisites and execution
1. Run `python generate_data.py` from the repository root.
2. Install packages with `pip install -r requirements.txt`.
3. Run this notebook from the repository root so relative paths work.

All data and policies are fictional. The LLM calls are handled by `mock_llm.py` locally; no API key is required.

## 0. Repository map and end-to-end flow

`generate_data.py` → `data/raw/*.csv` → audit → clean → `data/clean/*.csv` + `data/instamart.db` → RFM/segments + lapse model → privacy-safe ticket triage → policy retrieval + refund tools.

**Data-ingestion concept:** fetching data does not always mean calling an API. Here, the source is a local CSV export. In a production system, the same ingestion layer could use an approved API or warehouse connector, but credentials and network access are intentionally out of scope.

In [29]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root: repository root")
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: repository root
Root files: ['.gitignore', 'README.md', 'generate_data.py', 'mock_llm.py', 'requirements.txt', 'run_project.py']


## 1. Generate / fetch the raw data

The data generator is deterministic (fixed random seed). It writes customer, product, order, support-ticket, evaluation-question, agent-scenario inputs and policy text files.

Run it once when starting from a clean clone. Running it again recreates the same synthetic source exports.

In [30]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "generate_data.py")], check=True)
raw_dir = ROOT / "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

Raw files: ['agent_scenarios.csv', 'customers.csv', 'orders.csv', 'policy_eval.csv', 'products.csv', 'tickets.csv']


## 2. Read the exports and understand their shape

Use `keep_default_na=False` for ticket text so empty strings remain empty strings. This is important because empty tickets must be identified by an explicit guardrail rather than silently changing the input representation.

In [31]:
import pandas as pd
customers_raw = pd.read_csv(raw_dir / "customers.csv", keep_default_na=False)
products_raw = pd.read_csv(raw_dir / "products.csv", keep_default_na=False)
orders_raw = pd.read_csv(raw_dir / "orders.csv", keep_default_na=False)
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for name, df in [("customers", customers_raw), ("products", products_raw),
                 ("orders", orders_raw), ("tickets", tickets_raw)]:
    print(f"\\n{name}: shape={df.shape}")
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "blank_count": df.astype(str).apply(lambda col: col.str.strip().eq("").sum()),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }))

\ncustomers: shape=(1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,MALE
1,C0002,DELHI,2024-02-17,31,MALE
2,C0003,bengaluru,2025-02-18,18,F


,dtype,missing_count,blank_count,missing_pct
customer_id,str,0,0,0.0
city,str,0,0,0.0
signup_date,str,0,0,0.0
age,str,0,61,0.0
gender,str,0,30,0.0


\nproducts: shape=(40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


,dtype,missing_count,blank_count,missing_pct
product_id,str,0,0,0.0
product_name,str,0,0,0.0
category,str,0,0,0.0
price_inr,int64,0,0,0.0


... (2 outputs left out)

,dtype,missing_count,blank_count,missing_pct
order_id,str,0,0,0.0
customer_id,str,0,0,0.0
product_id,str,0,0,0.0
quantity,int64,0,0,0.0
order_ts,str,0,0,0.0
status,str,0,0,0.0
discount_pct,int64,0,0,0.0
order_amount,str,0,0,0.0
delivery_minutes,str,0,2541,0.0


\ntickets: shape=(164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


,dtype,missing_count,blank_count,missing_pct
ticket_id,str,0,0,0.0
customer_id,str,0,0,0.0
order_id,str,0,8,0.0
text,str,0,4,0.0
true_category,str,0,0,0.0


### Data-quality review questions
- Which columns are supposed to be unique keys?
- Are duplicates exact duplicates or multiple states of the same order?
- Which values are impossible (negative delivery duration, non-numeric money, implausible age)?
- Why should we distinguish an empty string from a missing value?
- What would go wrong if we modelled before cleaning?

In [32]:
from src.data_pipeline import audit, clean_data, validate
audit()


customers: shape=(1200, 5)
customer_id    str
city           str
signup_date    str
age            str
gender         str
dtype: object
             missing_count  missing_pct
customer_id              0          0.0
city                     0          0.0
signup_date              0          0.0
age                     61         5.08
gender                  30          2.5

products: shape=(40, 4)
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object
              missing_count  missing_pct
product_id                0          0.0
product_name              0          0.0
category                  0          0.0
price_inr                 0          0.0

orders: shape=(13084, 9)
order_id              str
customer_id           str
product_id            str
quantity            int64
order_ts              str
status                str
discount_pct        int64
order_amount          str
delivery_minutes      str
dtype: object
                 

## 3. Clean the data and write trusted outputs

The pipeline applies explicit rules:
- standardize city and gender;
- make impossible ages missing and fill with city median;
- parse timestamps, remove exact duplicates, then keep the latest row per order ID;
- coerce invalid amounts to missing and calculate replacements from product price × quantity × discount;
- set negative delivery times to missing and impute only completed orders using product-category medians;
- save clean CSVs and SQLite tables.

**Interpretation:** cancelled orders have no completed delivery duration, so this value should remain missing rather than be imputed as if delivery occurred.

In [33]:
clean = clean_data()
for name, df in clean.items():
    print(name, df.shape)
    display(df.head(3))
validate()

customers (1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,male
1,C0002,Delhi,2024-02-17,31,male
2,C0003,Bengaluru,2025-02-18,18,female


products (40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


orders (12309, 9)


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
827,ORD-100001,C0001,P001,1,2025-09-16 08:07:21,completed,15,109.65,20
4294,ORD-100002,C0001,P008,1,2025-09-03 07:34:01,completed,40,25.20,23
9973,ORD-100003,C0001,P008,1,2025-04-08 17:08:14,completed,0,42.00,23


ALL CHECKPOINTS PASSED


## 4. Query the SQLite database

The clean tables are persisted in `data/instamart.db`. SQLite gives us a portable analytical database that can be queried without setting up a server.

Run the supplied SQL examples and compare SQL totals to pandas results.

In [34]:
import sqlite3
db_path = ROOT / "data" / "instamart.db"
with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
    display(tables)
    q2 = pd.read_sql_query("""
        SELECT c.city, p.category, ROUND(SUM(o.order_amount), 2) AS revenue
        FROM orders o
        JOIN customers c USING(customer_id)
        JOIN products p USING(product_id)
        WHERE o.status = 'completed'
        GROUP BY c.city, p.category
        ORDER BY c.city, p.category
    """, con)
display(q2)
print("SQL grand total INR:", round(q2.revenue.sum(), 2))

,name
0,customers
1,orders
2,products


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


SQL grand total INR: 2998668.8


## 5. Part B — customer segments and lapse prediction

RFM means:
- **Recency:** days since the most recent completed order at the snapshot date.
- **Frequency:** number of distinct completed orders.
- **Monetary:** total spend across completed orders.

The model module scales RFM for K-means, compares K values, uses PCA to inspect compressed variance, and builds a lapse model. The lapse model's input features use only orders before 2025-09-01; September is reserved to define the label. The scaler and encoder are fitted inside a training pipeline, not on the test data.

In [35]:
from src.customer_models import rfm_and_segments, build_lapse_model
rfm, k_diagnostics, segment_profile, pca_variance = rfm_and_segments(k=4)
display(k_diagnostics.round(3))
display(segment_profile.round(2))
print("PCA explained variance ratio:", pca_variance, "total:", pca_variance.sum())
display(rfm.head())
lapse_summary = build_lapse_model()

lapse_summary_display = dict(lapse_summary)
lapse_summary_display["model_path"] = "outputs/lapse_model.joblib"

lapse_summary_display

,k,inertia,silhouette
0,2,1820.761,0.465
1,3,1268.235,0.445
2,4,743.071,0.526
3,5,617.367,0.480
4,6,503.171,0.482
5,7,408.460,0.416
6,8,366.503,0.412


,segment_id,size,recency,frequency,monetary
0,0,436,28.86,3.77,1057.62
1,1,138,24.12,6.77,7129.70
2,2,419,8.77,19.40,3410.02
3,3,182,109.04,2.20,685.99


PCA explained variance ratio: [0.63427875 0.21803762] total: 0.8523163667502166


,customer_id,frequency,monetary,recency,segment_id,segment_name,is_premium
0,C0001,4,512.85,14,0,Occasional,False
1,C0002,7,4644.90,28,1,Premium,True
2,C0003,22,3644.85,7,2,Frequent,False
3,C0004,5,369.00,0,0,Occasional,False
4,C0005,2,324.00,147,3,Dormant,False


{'cohort_rows': 1156,
 'lapsed': 443,
 'baseline_accuracy': 0.6163793103448276,
 'models': {'logistic': {'best_params': {'model__C': 100},
   'cv_f1': np.float64(0.650552962900789),
   'precision': 0.6444444444444445,
   'recall': 0.651685393258427,
   'f1': 0.6480446927374302,
   'confusion_matrix': [[111, 32], [31, 58]]},
  'forest': {'best_params': {'model__max_depth': 8,
    'model__min_samples_leaf': 10},
   'cv_f1': np.float64(0.6865517479789224),
   'precision': 0.6333333333333333,
   'recall': 0.6404494382022472,
   'f1': 0.6368715083798883,
   'confusion_matrix': [[110, 33], [32, 57]]}},
 'oversampling': {'class_counts_before': {0: 570, 1: 354},
  'class_counts_after': {0: 570, 1: 570},
  'training_rows_after_oversampling': 1140,
  'precision': 0.5891472868217055,
  'recall': 0.8539325842696629,
  'f1': 0.6972477064220184,
  'confusion_matrix': [[90, 53], [13, 76]]},
 'feature_insights': {'random_forest_top_5': [{'feature': 'num__monetary',
    'importance': 0.2804279732344776

### Leakage prevention and interpretation
Review the following leakage considerations:
1. Which dates can enter the lapse-model features?
2. Why must September orders not enter the features?
3. Why is a Premium-only retention list labelled *illustrative* if the Premium flag comes from whole-window RFM?
4. Why can accuracy be misleading when a substantial minority of customers lapse?

## 6. Part C — support ticket triage

This pipeline demonstrates:
1. input guardrails for empty and overlong tickets;
2. ordered PII redaction (email → phone → order ID);
3. retry logic for simulated transient errors;
4. strict category validation and a keyword fallback;
5. token budgeting and human-review routing.

The exact user message sent to the mock classifier is the redacted ticket text only.

In [36]:
from src.triage import run_pipeline, FEW_SHOT_PROMPT, ZERO_SHOT_PROMPT

triage_results, triage_summary, confusion_table = run_pipeline(
    prompt=FEW_SHOT_PROMPT,
    token_budget=100000
)

print(triage_summary)
display(confusion_table)
display(triage_results.head(10))
display(
    triage_results.status.value_counts()
    .rename_axis("status")
    .reset_index(name="tickets")
)

RETRY ticket_id=T0002 retry_number=0 wait_seconds=0.0530
RETRY ticket_id=T0116 retry_number=0 wait_seconds=0.0453
RETRY ticket_id=T0153 retry_number=0 wait_seconds=0.0484
RETRY ticket_id=T0039 retry_number=0 wait_seconds=0.0478
RETRY ticket_id=T0152 retry_number=0 wait_seconds=0.0542
RETRY ticket_id=T0043 retry_number=0 wait_seconds=0.0535
RETRY ticket_id=T0111 retry_number=0 wait_seconds=0.0562
RETRY ticket_id=T0155 retry_number=0 wait_seconds=0.0461
RETRY ticket_id=T0100 retry_number=0 wait_seconds=0.0503
RETRY ticket_id=T0068 retry_number=0 wait_seconds=0.0454
RETRY ticket_id=T0135 retry_number=0 wait_seconds=0.0477
RETRY ticket_id=T0009 retry_number=0 wait_seconds=0.0513
RETRY ticket_id=T0067 retry_number=0 wait_seconds=0.0453
RETRY ticket_id=T0106 retry_number=0 wait_seconds=0.0475
RETRY ticket_id=T0097 retry_number=0 wait_seconds=0.0531
RETRY ticket_id=T0003 retry_number=0 wait_seconds=0.0518
RETRY ticket_id=T0019 retry_number=0 wait_seconds=0.0478
RETRY ticket_id=T0086 retry_num

category,billing,delivery,fraud,technical
true_category,,,,
billing,38,0,0,0
delivery,0,36,0,0
fraud,0,0,34,0
technical,0,0,0,39


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,Dormant,HIGH,security_team,125
1,T0073,C0080,classified_json,billing,Frequent,NORMAL,support_queue,121
2,T0095,C1087,classified_json,technical,Occasional,NORMAL,support_queue,123
3,T0159,C1172,skipped_empty,,Frequent,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,Occasional,HIGH,security_team,125
5,T0124,C0139,classified_json,fraud,Frequent,HIGH,security_team,125
6,T0061,C0812,classified_json,billing,Frequent,NORMAL,support_queue,117
7,T0118,C0355,classified_json,delivery,Frequent,NORMAL,support_queue,120
8,T0037,C1156,classified_json,billing,Premium,HIGH,premium_desk,121
9,T0002,C0881,classified_json,delivery,Occasional,NORMAL,support_queue,119


,status,tickets
0,classified_json,135
1,classified_keyword,12
2,flagged_invalid_output,9
3,skipped_empty,4
4,flagged_long,4


### Privacy and safety review
- Redaction is a defence-in-depth step, not a guarantee that every sensitive detail is removed.
- Why are invalid categories sent to human review instead of guessed?
- Why should fraud override ordinary customer-segment routing?
- Why do empty/long tickets need to be blocked before any model call?

## 7. Part D — policy RAG

The retrieval layer chunks policy documents, fits a TF-IDF vectorizer and ranks chunks by cosine similarity. It passes retrieved chunks to the offline policy assistant, which should answer only from the context and fall back when a question is not covered.

In [37]:
from src.policy_rag import PolicyRAG, evaluate
rag = PolicyRAG(chunk_size=40, overlap=10)
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())
print(rag.answer("What delivery time makes a completed order eligible for a full refund?"))
policy_eval = evaluate(rag)
display(policy_eval)
print("Expected-document hits:", int(policy_eval.retrieved_doc.sum()))
print("Expected-fact hits:", int(policy_eval.fact_found.sum()))
print("Fallbacks:", int(policy_eval.fallback.sum()))

# Required comparison: repeat the evaluation with no chunk overlap.
rag_no_overlap = PolicyRAG(chunk_size=40, overlap=0)
policy_eval_no_overlap = evaluate(rag_no_overlap)
display(policy_eval_no_overlap)

answerable_no_overlap = policy_eval_no_overlap[
    (policy_eval_no_overlap.expected_doc != "NONE")
]
print(
    "Answerable questions failing expected-fact check (overlap=0):",
    int((~answerable_no_overlap.fact_found).sum())
)

Policy chunks: 15


,source,text
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...
1,delivery_policy.txt,"of INR 25 is charged on orders below INR 199, ..."
2,delivery_policy.txt,60 minutes and the app shows a rain notice. A ...
3,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...
4,escalation_policy.txt,"1 hour, and all other tickets within 24 hours...."


{'answer': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.', 'sources': ['refund_policy.txt', 'privacy_policy.txt'], 'retrieved': [{'source': 'refund_policy.txt', 'score': 0.35464416504570484, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.31425507733397284, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'privacy_policy.txt', 'score': 0.15759666156595936, 'text': 'Delivery addresses are deleted from active systems 180 days after the last order. A customer can ask for account deletion and the request is c

,question,expected_doc,retrieved_doc,fact_found,fallback
0,What delivery time makes a completed order eli...,refund_policy.txt,True,True,False
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,False
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,False
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,False
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,False
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,False
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,False
7,How soon is money returned when a payment fail...,payment_policy.txt,True,True,False
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,False
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,False


Expected-document hits: 12
Expected-fact hits: 12
Fallbacks: 2


,question,expected_doc,retrieved_doc,fact_found,fallback
0,What delivery time makes a completed order eli...,refund_policy.txt,True,False,False
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,False
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,False
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,False
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,False
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,False
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,False
7,How soon is money returned when a payment fail...,payment_policy.txt,True,False,False
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,False
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,False


Answerable questions failing expected-fact check (overlap=0): 2


## 8. Part D — refund tools and safe stopping

The refund agent demonstrates a tool loop: look up an order in SQLite, retrieve policy, calculate the eligible refund, then issue it or escalate to a human. The refund ledger is a local JSONL simulation; no real financial transaction is performed.

Before running agent demonstrations, inspect `src/refund_agent.py` and trace how each tool name maps to a Python function.

In [38]:
from run_project import run_part_d_agent

# Executes S1-S7, including S1 checkpoint/crash/resume,
# plus token-limit and timeout fallback demonstrations.
run_part_d_agent()

\n=== PART D: REFUND AGENT SCENARIOS ===
S1: simulated crash after tool 2; resuming checkpoint.
S1: status=REFUND, model_calls=5, tokens=900
S2: status=REFUND, model_calls=5, tokens=900
S3: status=ESCALATE, model_calls=5, tokens=887
S4: status=DENIED, model_calls=4, tokens=605
S5: status=DENIED, model_calls=3, tokens=337
S6: status=NOT_FOUND, model_calls=2, tokens=72
S7: status=FALLBACK_MAX_ITERATIONS, model_calls=6, tokens=3000
\nScenario summary:
scenario_id customer_id expected_outcome actual_outcome           actual_status  outcome_match  expected_refund_inr  actual_refund_inr  llm_calls  tokens_used                                                                             tools_executed
         S1       C0007           REFUND         REFUND                  REFUND           True               128.25             128.25          5          900                                 [get_order, search_policy, calculate_refund, issue_refund]
         S2       C0008           REFUND       

## 9. Final generated artifacts and reproducibility check

Review the CSVs and model file generated by the pipeline. **Reproducibility check:** remove the generated `data/clean/`, `data/instamart.db`, and `outputs/` artifacts, then rerun the documented commands to verify that the project can be rebuilt from source.

**Important:** This project is a local implementation using synthetic data and an offline mock LLM. If the course grades exact numeric checkpoints from the original brief, run every checkpoint and reconcile differences; do not hardcode expected answers.

In [39]:
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\n{folder.relative_to(ROOT)}")

    if folder.exists():
        files = [p for p in folder.iterdir() if p.is_file()]
        print("File count:", len(files))
    else:
        print("Directory not created")

print("Database exists:", db_path.exists())


data\clean
File count: 3

outputs
File count: 8

state
File count: 112
Database exists: True


### Environment Verification

Before executing the project notebook, verify that it is running with the correct Python environment and compatible library versions.

This check helps ensure consistency between the environment used to train the machine learning model and the environment used to load and execute it.

The following code displays:
- **Python Executable:** Confirms the active Python interpreter and virtual environment.
- **Scikit-learn Version:** Checks the installed scikit-learn version used for model training and loading.
- **Joblib Version:** Checks the installed joblib version used for saving and loading the trained model.

This verification helps identify environment mismatches and prevent potential model compatibility issues.

In [40]:
import sys
import sklearn
import joblib

print("Python version:", sys.version.split()[0])
print("Virtual environment:", sys.prefix != sys.base_prefix)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

Python version: 3.12.0
Virtual environment: True
Scikit-learn: 1.9.1
Joblib: 1.6.0
